# BCT-AI-Pharma Results Visualization

Loads `results/simulation_results.csv` and `results/performance_metrics.json` (produced by `python main.py` or `python evaluation/run_simulation.py`) and visualizes the computed metrics. Release 1.1.0 removed the comparisons with conference-reported target values; those are reference values, not computed results (see `historical/bcca2026_reference.py`).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = REPO_ROOT / "results"

with open(RESULTS_DIR / "performance_metrics.json") as f:
    metrics = json.load(f)

sim_results = pd.read_csv(RESULTS_DIR / "simulation_results.csv")
sim_results.head()

## 1. Blockchain performance under varying transaction load (Table III)

In [ ]:
blockchain_df = pd.DataFrame(metrics["blockchain_performance"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(blockchain_df["tps_target"], blockchain_df["avg_latency_ms"], marker="o", label="Avg latency")
axes[0].plot(blockchain_df["tps_target"], blockchain_df["p95_latency_ms"], marker="s", label="P95 latency")
axes[0].set_xlabel("Offered load (TPS)")
axes[0].set_ylabel("Latency (ms)")
axes[0].set_title("Latency vs. throughput")
axes[0].legend()

axes[1].plot(blockchain_df["tps_target"], blockchain_df["success_rate_pct"], marker="o", color="green")
axes[1].set_xlabel("Offered load (TPS)")
axes[1].set_ylabel("Success rate (%)")
axes[1].set_title("Transaction success rate vs. throughput")
plt.tight_layout()
plt.show()

## 2. LSTM demand forecasting / delay prediction (computed metrics)

In [ ]:
# Release 1.1.0: conference-reported "paper target" values are no longer plotted
# here (they are reference values, not computed results).
lstm_actual = pd.Series(metrics["lstm"]["metrics"], name="this_run")
lstm_actual.plot(kind="bar", figsize=(7, 4))
plt.title("LSTM: computed metrics (this run)")
plt.tight_layout()
plt.show()
lstm_actual.to_frame()

## 3. CNN packaging verification performance by tamper class

In [ ]:
# Release 1.1.0: computed metrics only (no conference-reported values).
cnn_actual = metrics["cnn"]["metrics"]
rows = [{"class": cls, "f1": vals["f1"]} for cls, vals in cnn_actual.items()
        if isinstance(vals, dict) and "f1" in vals]
cnn_f1 = pd.DataFrame(rows)
plt.figure(figsize=(9, 4))
plt.bar(cnn_f1["class"], cnn_f1["f1"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("F1 score")
plt.title("CNN F1 by class (this run)")
plt.tight_layout()
plt.show()
cnn_f1

## 4. Isolation Forest cold chain anomaly detection by anomaly type

In [ ]:
# Release 1.1.0: computed metrics only (no conference-reported values).
if_actual = metrics["isolation_forest"]["metrics"]
rows = [{"anomaly_type": k, "detection": v.get("detection", float("nan")), "fpr": v.get("fpr", float("nan"))}
        for k, v in if_actual.items() if isinstance(v, dict)]
if_df = pd.DataFrame(rows)
plt.figure(figsize=(9, 4))
plt.bar(if_df["anomaly_type"], if_df["detection"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Detection rate")
plt.title("Isolation Forest detection rate by anomaly type (this run)")
plt.tight_layout()
plt.show()
if_df

## 5. Product Trust Score: distribution by drug class and weight sensitivity

In [ ]:
pts_scores = metrics["pts"]["sample_scores"]
pts_df = pd.DataFrame(pts_scores).T
pts_df.index.name = "drug_class"
pts_df

In [ ]:
# Nested {scenario: {component: [rows]}}; only "minor_excursion" / "temperature_compliance"
# is calibrated against the paper's one published sensitivity figure (Section 3.5.4,
# ~-0.08 PTS per +10-point weight shift) -- see pts/pts_sensitivity_analysis.py docstring.
scenario_curves = metrics["pts"]["sensitivity_curves"]
fig, axes = plt.subplots(1, len(scenario_curves), figsize=(6 * len(scenario_curves), 4.5), sharey=True)
for ax, (scenario, curves) in zip(axes, scenario_curves.items()):
    for component, rows in curves.items():
        df = pd.DataFrame(rows)
        ax.plot(df["delta_pct"], df["pts"], marker="o", label=component)
    ax.set_xlabel("Weight variation (percentage points)")
    ax.set_title(scenario.replace("_", " ").title())
    ax.legend(fontsize=8)
axes[0].set_ylabel("PTS")
fig.suptitle("Class A sensitivity: PTS vs. weight re-allocation, by excursion scenario")
plt.tight_layout()
plt.show()

## 6. BCCA 2026 conference-reported reference values (removed from this notebook)

Release 1.1.0 removed the bar charts that plotted the "Table VI baseline comparison" here.
Those values (96.8 / 78.2 / 41.7 / 23.4% and the associated localization times) are BCCA 2026
conference-reported reference values read from configuration. They are not computed by this repository,
so they are no longer shown in evidentiary plots. To display them, clearly labelled, run
`python -m historical.bcca2026_reference`.


## 7. Statistical validation across independent runs

In [ ]:
sv = metrics["statistical_validation"]
print("Counterfeit detection paired t-test:", sv["counterfeit_detection_ttest"])
print("Recall efficiency paired t-test:", sv["recall_efficiency_ttest"])
print("95% CI (counterfeit detection):", sv["confidence_intervals_95"]["bct_ai_counterfeit_detection"])
print("95% CI (recall efficiency):", sv["confidence_intervals_95"]["bct_ai_recall_efficiency"])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sim_results["run_id"], sim_results["counterfeit_detection_rate"], marker="o", label="Counterfeit detection")
ax.plot(sim_results["run_id"], sim_results["recall_efficiency"], marker="s", label="Recall efficiency")
ax.set_xlabel("Run")
ax.set_ylabel("Rate")
ax.set_title("Per-run metrics across 10 independent simulation runs")
ax.legend()
plt.tight_layout()
plt.show()